In [1]:
# Cell 1: tools
# Programmable 3-mode Fock-state interferometer simulator
#
# Physics:
#   - Passive 3-mode interferometer
#   - Fock inputs n_i in {0,1}
#   - Three MZI layers: (0,1), (1,2), (0,1)
#   - One tunable internal phase per MZI
#   - Photon-number-resolved output probabilities
#   - Total photon number is conserved
#
# Convention:
#   MZI(phi) = B P(phi) B using Gerry--Knight balanced beamsplitter
#   B = 1/sqrt(2) [[1, i], [i, 1]]
#   P(phi) = diag(exp(i phi), 1)
#
# Up to a global phase, this gives the real effective block
#   [[ sin(phi/2),  cos(phi/2)],
#    [ cos(phi/2), -sin(phi/2)]]
#
# For Fock-state passive linear optics, transition amplitudes are given by
# matrix permanents of repeated-row/repeated-column submatrices.

import math
import itertools
import numpy as np
import matplotlib.pyplot as plt


def mzi_block(phi: float, use_complex_gerry_knight: bool = False) -> np.ndarray:
    """
    Return a 2x2 one-parameter MZI matrix.

    If use_complex_gerry_knight=False, return the effective real MZI block
    obtained from B P(phi) B after dropping a global phase.

    If use_complex_gerry_knight=True, return the full complex Gerry--Knight
    convention:
        B = 1/sqrt(2) [[1, i], [i, 1]]
        P = diag(exp(i phi), 1)
        MZI = B @ P @ B

    Both give identical output probabilities for this simulator.
    """
    phi = float(phi)

    if use_complex_gerry_knight:
        B = (1.0 / np.sqrt(2.0)) * np.array(
            [[1.0, 1.0j],
             [1.0j, 1.0]],
            dtype=complex,
        )
        P = np.array(
            [[np.exp(1.0j * phi), 0.0],
             [0.0, 1.0]],
            dtype=complex,
        )
        return B @ P @ B

    return np.array(
        [
            [np.sin(phi / 2.0),  np.cos(phi / 2.0)],
            [np.cos(phi / 2.0), -np.sin(phi / 2.0)],
        ],
        dtype=complex,
    )


def embed_two_mode(block_2x2: np.ndarray, modes: tuple[int, int], n_modes: int = 3) -> np.ndarray:
    """
    Embed a 2x2 block into an n_modes x n_modes identity matrix.

    modes gives the two mode indices affected by the block.
    """
    i, j = modes
    U = np.eye(n_modes, dtype=complex)

    U[i, i] = block_2x2[0, 0]
    U[i, j] = block_2x2[0, 1]
    U[j, i] = block_2x2[1, 0]
    U[j, j] = block_2x2[1, 1]

    return U


def build_three_mode_interferometer(
    phi_01_a: float,
    phi_12: float,
    phi_01_b: float,
    use_complex_gerry_knight: bool = False,
) -> np.ndarray:
    """
    Build the 3-mode interferometer.

    Layer 0: MZI on modes (0,1)
    Layer 1: MZI on modes (1,2)
    Layer 2: MZI on modes (0,1)

    If column vectors transform as y = U x, then the full transformation is
        U = U_layer2 @ U_layer1 @ U_layer0.
    """
    U0 = embed_two_mode(
        mzi_block(phi_01_a, use_complex_gerry_knight=use_complex_gerry_knight),
        modes=(0, 1),
        n_modes=3,
    )
    U1 = embed_two_mode(
        mzi_block(phi_12, use_complex_gerry_knight=use_complex_gerry_knight),
        modes=(1, 2),
        n_modes=3,
    )
    U2 = embed_two_mode(
        mzi_block(phi_01_b, use_complex_gerry_knight=use_complex_gerry_knight),
        modes=(0, 1),
        n_modes=3,
    )

    return U2 @ U1 @ U0


def permanent(A: np.ndarray) -> complex:
    """
    Compute the permanent of a small square matrix by direct permutation sum.

    This is intentionally simple because the simulator only has at most
    three input photons.
    """
    A = np.asarray(A, dtype=complex)
    n_rows, n_cols = A.shape

    if n_rows != n_cols:
        raise ValueError("Permanent requires a square matrix.")

    n = n_rows

    if n == 0:
        return 1.0 + 0.0j

    total = 0.0 + 0.0j
    for perm in itertools.permutations(range(n)):
        prod = 1.0 + 0.0j
        for i, j in enumerate(perm):
            prod *= A[i, j]
        total += prod

    return total


def weak_compositions(total: int, parts: int) -> list[tuple[int, ...]]:
    """
    Enumerate all nonnegative integer tuples of length 'parts' summing to total.
    """
    if parts <= 0:
        return []

    if parts == 1:
        return [(total,)]

    out = []
    for first in range(total + 1):
        for rest in weak_compositions(total - first, parts - 1):
            out.append((first,) + rest)
    return out


def factorial_product(values: tuple[int, ...]) -> int:
    """
    Product of factorials for a tuple of occupation numbers.
    """
    prod = 1
    for v in values:
        prod *= math.factorial(int(v))
    return prod


def repeated_indices(occupations: tuple[int, ...]) -> list[int]:
    """
    Expand occupation tuple into repeated mode indices.

    Example:
        (2,0,1) -> [0,0,2]
    """
    idx = []
    for mode, count in enumerate(occupations):
        idx.extend([mode] * int(count))
    return idx


def fock_transition_amplitude(
    U: np.ndarray,
    n_in: tuple[int, int, int],
    n_out: tuple[int, int, int],
) -> complex:
    """
    Passive-linear-optics Fock transition amplitude.

    Amplitude:
        <n_out| U |n_in>
        = permanent(U_sub) / sqrt(prod_i n_in_i! prod_j n_out_j!)

    U_sub is formed by repeating columns according to n_in and rows according
    to n_out.
    """
    if sum(n_in) != sum(n_out):
        return 0.0 + 0.0j

    N = sum(n_in)

    if N == 0:
        return 1.0 + 0.0j

    in_cols = repeated_indices(n_in)
    out_rows = repeated_indices(n_out)

    U_sub = U[np.ix_(out_rows, in_cols)]

    denom = np.sqrt(factorial_product(n_in) * factorial_product(n_out))
    return permanent(U_sub) / denom


def fock_output_distribution(
    U: np.ndarray,
    n_in: tuple[int, int, int],
) -> dict[tuple[int, int, int], float]:
    """
    Return photon-number-resolved output probabilities for all output
    patterns conserving total photon number.
    """
    total_photons = sum(n_in)
    outcomes = weak_compositions(total_photons, 3)

    probs = {}
    for n_out in outcomes:
        amp = fock_transition_amplitude(U, n_in=n_in, n_out=n_out)
        probs[n_out] = float(np.abs(amp) ** 2)

    # Numerical cleanup and normalization.
    total_prob = sum(probs.values())
    if total_prob > 0:
        probs = {k: max(0.0, v / total_prob) for k, v in probs.items()}

    return probs


def ket_label(occupation: tuple[int, int, int]) -> str:
    """
    Format occupation tuple as a ket label.
    """
    return r"$|" + ",".join(str(int(x)) for x in occupation) + r"\rangle$"


def unitary_error(U: np.ndarray) -> float:
    """
    Return ||U^\dagger U - I||_F as a quick unitarity diagnostic.
    """
    n = U.shape[0]
    return float(np.linalg.norm(U.conj().T @ U - np.eye(n), ord="fro"))


def print_unitary_summary(U: np.ndarray) -> None:
    """
    Print compact numerical unitary summary.
    """
    np.set_printoptions(precision=3, suppress=True)
    print("3-mode interferometer U:")
    print(U)
    print(f"\nUnitarity check ||U†U - I||_F = {unitary_error(U):.3e}")

<>:253: SyntaxWarning: invalid escape sequence '\d'
<>:253: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_5333/3473451365.py:253: SyntaxWarning: invalid escape sequence '\d'
  """


In [2]:
# Cell 2: UI and output

import ipywidgets as widgets
from IPython.display import display, clear_output

# Input mode selectors: Fock occupation 0 or 1 in each input mode.
n0_widget = widgets.ToggleButtons(
    options=[0, 1],
    value=1,
    description="Input n0",
    button_style="",
)

n1_widget = widgets.ToggleButtons(
    options=[0, 1],
    value=1,
    description="Input n1",
    button_style="",
)

n2_widget = widgets.ToggleButtons(
    options=[0, 1],
    value=0,
    description="Input n2",
    button_style="",
)

input_row = widgets.HBox([n0_widget, n1_widget, n2_widget])

# Phase sliders.
phi_01_a_widget = widgets.FloatSlider(
    value=np.pi / 2,
    min=0.0,
    max=2.0 * np.pi,
    step=0.01,
    description=r"phi 01 A",
    continuous_update=False,
    readout_format=".2f",
)

phi_12_widget = widgets.FloatSlider(
    value=np.pi / 2,
    min=0.0,
    max=2.0 * np.pi,
    step=0.01,
    description=r"phi 12",
    continuous_update=False,
    readout_format=".2f",
)

phi_01_b_widget = widgets.FloatSlider(
    value=np.pi / 2,
    min=0.0,
    max=2.0 * np.pi,
    step=0.01,
    description=r"phi 01 B",
    continuous_update=False,
    readout_format=".2f",
)

phase_row = widgets.VBox([phi_01_a_widget, phi_12_widget, phi_01_b_widget])

# Optional convention toggle.
complex_convention_widget = widgets.Checkbox(
    value=False,
    description="Use full complex Gerry--Knight MZI convention",
    indent=False,
)

output = widgets.Output()


def update_plot(*args):
    """
    Update the PNR output histogram.
    """
    with output:
        clear_output(wait=True)

        n_in = (
            int(n0_widget.value),
            int(n1_widget.value),
            int(n2_widget.value),
        )

        total_photons = sum(n_in)

        U = build_three_mode_interferometer(
            phi_01_a=phi_01_a_widget.value,
            phi_12=phi_12_widget.value,
            phi_01_b=phi_01_b_widget.value,
            use_complex_gerry_knight=complex_convention_widget.value,
        )

        probs = fock_output_distribution(U, n_in=n_in)

        # Sort outcomes lexicographically for stable display.
        outcomes = sorted(probs.keys(), reverse=True)
        labels = [ket_label(o) for o in outcomes]
        values = [probs[o] for o in outcomes]

        print(
            f"Input state: {ket_label(n_in)}    "
            f"Total photons: N = {total_photons}"
        )
        print(
            "Photon-number constraint: showing only outcomes with "
            f"n0' + n1' + n2' = {total_photons}"
        )
        print(f"Probability normalization: {sum(values):.6f}")
        print(f"Unitarity check ||U†U - I||_F = {unitary_error(U):.3e}")

        fig_width = max(6.0, 0.65 * len(labels))
        plt.figure(figsize=(fig_width, 4.2))
        plt.bar(range(len(values)), values)
        plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
        plt.ylim(0.0, 1.05)
        plt.ylabel("Probability")
        plt.xlabel("PNR output pattern")
        plt.title(
            "3-mode programmable Fock-state interferometer\n"
            f"{ket_label(n_in)} input; photon-number-conserving PNR outputs"
        )
        plt.tight_layout()
        plt.show()

        # Compact probability table.
        print("\nOutput probabilities:")
        for outcome, p in zip(outcomes, values):
            print(f"  {ket_label(outcome):>16s} : {p:.6f}")


# Attach callbacks.
for w in [
    n0_widget,
    n1_widget,
    n2_widget,
    phi_01_a_widget,
    phi_12_widget,
    phi_01_b_widget,
    complex_convention_widget,
]:
    w.observe(update_plot, names="value")

ui = widgets.VBox(
    [
        widgets.HTML("<h3>Programmable 3-mode Fock-state interferometer</h3>"),
        widgets.HTML(
            "Select Fock inputs <code>n0,n1,n2 ∈ {0,1}</code>, tune the three "
            "MZI internal phases, and view all photon-number-conserving "
            "PNR output probabilities."
        ),
        widgets.HTML("<b>Input occupations</b>"),
        input_row,
        widgets.HTML("<b>MZI internal phases</b>"),
        phase_row,
        complex_convention_widget,
        output,
    ]
)

display(ui)
update_plot()